This notebook contains the code to generate all plots on the IL-2 mutants with enhanced IL-2RG binding

In [112]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
from scipy.optimize import curve_fit
from src.simulate_func import hill_func
from collections import Counter
from matplotlib.lines import Line2D
import re
from scipy.stats import ttest_ind

## Get dose response curves from experiments 1 and 2

In [73]:
df_concentration = pd.read_csv("data/signaling/IL2_mutants/IL2_concentrations.csv",index_col=0)
df_concentration = df_concentration.loc[['WT__1', 'WT__2', 'Q22R__1',
       'Q22R__2', 'S130L__1', 'S130L__2', 'S130M__1', 'S130M__2',
       'Q22R_S130L__1', 'Q22R_S130L__2', 'Q22R_S130M__1', 'Q22R_S130M__2',
       'E15K__1', 'E15K__2', 'K49R__1', 'K49R__2',
       'E15K_K49R__1', 'E15K_K49R__2',
       'Q126A__1', 'Q126A__2', 'T123A__1',
       'T123A__2', 'N119E__1', 'N119E__2']]
# Normalization by western blot of the concetrations
norm_factors = np.array([1,1,2.12430489,2.12430489,2.64633585,2.64633585,3.16772925,3.16772925,2.85771969,2.85771969,
       2.84059434,2.84059434,1.38694081,1.38694081,2.05837141,2.05837141,1.17013778,1.17013778,
       1.73642546,1.73642546,3.44223072,3.44223072,2.9376616,2.9376616])
df_concentration = df_concentration.div(norm_factors, axis=0)
data = "HEK_RBRG_60"
df_signaling = pd.read_csv("data/signaling/IL2_mutants/"+data+".csv",index_col=0)
background = df_signaling.loc["Background"].melt()["value"].mean()
df_signaling = df_signaling.loc[df_concentration.index] - background
df_concentration = df_concentration[df_concentration.columns[:-1]]
df_signaling = df_signaling[df_signaling.columns[:-1]]
condition_names = [re.sub(r"__\d+$", "", str(idx)) for idx in df_signaling.index]

# Mean and std dev across replicates
conc_mean = df_concentration.groupby(condition_names).mean()
conc_std  = df_concentration.groupby(condition_names).std(ddof=0)

sig_mean = df_signaling.groupby(condition_names).mean()
sig_std  = df_signaling.groupby(condition_names).std(ddof=0)

# Plot order
plot_order = [
    'WT',
    'S130L',
    'S130M',
    'K49R',
    'T123A',
    'N119E',
    'Q22R',
    'Q22R_S130L',
    'Q22R_S130M',
    'E15K',
    'E15K_K49R',
    'Q126A',
]

# Keep only conditions that are actually present
plot_order = [condition for condition in plot_order if condition in sig_mean.index]
# Force each mutant to have its own color
cmap = plt.get_cmap("tab20", len(plot_order))
colors = {
    condition: cmap(i)
    for i, condition in enumerate(plot_order)
}

# Force each mutant to have its own marker shape
marker_shapes = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">", "h", "d"]
markers = {
    condition: marker_shapes[i % len(marker_shapes)]
    for i, condition in enumerate(plot_order)
}

df_EC50_RBRG_exp1 = pd.DataFrame(columns=plot_order, index=[1, 2])
fig, ax = plt.subplots(1, 1, figsize=(9, 8), dpi=400)
for condition in plot_order:
    # Save EC50s
    x = df_concentration.loc[condition+"__1"].values
    y = df_signaling.loc[condition+"__1"].values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_1, cov = curve_fit(hill_func, x[3:], y[3:]/np.max(y[3:])*100, bounds=([95, 1e-14], [105, 1e-12]), p0=[100, 1e-13])
    x = df_concentration.loc[condition+"__2"].values
    y = df_signaling.loc[condition+"__2"].values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_2, cov = curve_fit(hill_func, x[3:], y[3:]/np.max(y[3:])*100, bounds=([95, 1e-14], [105, 1e-12]), p0=[100, 1e-13])
    df_EC50_RBRG_exp1[condition] = [fit_1[1], fit_2[1]]

    # Plot
    x = conc_mean.loc[condition].values
    y = sig_mean.loc[condition].values
    yerr = sig_std.loc[condition].values
    ax.errorbar(
        x,
        y/sig_mean.loc['WT'].max()*100,
        yerr=yerr/sig_mean.loc['WT'].max()*100,
        marker=markers[condition],
        linestyle="-",
        label=condition,
        color=colors[condition],
        capsize=7,
        linewidth=3,
        elinewidth=3,
        capthick=3,
        markersize=9,
        markeredgewidth=2,
    )
plt.xlim([5e-15, 1e-9])
ax.legend(
    fontsize=17,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0,
)
plt.xscale('log')
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.title("HEK CD25⁻/CD122⁺/CD132⁺", fontsize=25)
plt.ylabel("IL2 signaling (%)", fontsize=25)
plt.xlabel('IL-2 (M)', fontsize=25)
plt.savefig('figures/IL2_mutants/HEK_RBRG_60_exp1.pdf', transparent=True, bbox_inches="tight")
plt.close()

In [74]:
data = "HEK_RARBRG_60"
df_signaling = pd.read_csv("data/signaling/IL2_mutants/"+data+".csv",index_col=0)
background = df_signaling.loc["Background"].melt()["value"].mean()
df_signaling = df_signaling.loc[df_concentration.index] - background
df_signaling = df_signaling.apply(pd.to_numeric)
df_signaling = df_signaling[df_signaling.columns[:-1]]
# df_concentration = df_concentration.div(1/norm_factors, axis=0)
# Check that both dataframes match
if not df_concentration.index.equals(df_signaling.index):
    raise ValueError("df_concentration and df_signaling must have the same index.")

if not df_concentration.columns.equals(df_signaling.columns):
    raise ValueError("df_concentration and df_signaling must have the same columns.")

# Extract mutant/condition name from replicate names
# Example: WT__1 -> WT, Q22R_S130L__2 -> Q22R_S130L
def get_condition_name(index_name):
    return re.sub(r"__\d+$", "", str(index_name))

condition_names = [get_condition_name(idx) for idx in df_signaling.index]

# Mean and std dev across replicates
conc_mean = df_concentration.groupby(condition_names).mean()
conc_std  = df_concentration.groupby(condition_names).std(ddof=0)

sig_mean = df_signaling.groupby(condition_names).mean()
sig_std  = df_signaling.groupby(condition_names).std(ddof=0)

# Plot order
plot_order = [
    'WT',
    'S130L',
    'S130M',
    'K49R',
    'T123A',
    'N119E',
    'Q22R',
    'Q22R_S130L',
    'Q22R_S130M',
    'E15K',
    'E15K_K49R',
    'Q126A',
]

# Keep only conditions that are actually present
plot_order = [condition for condition in plot_order if condition in sig_mean.index]
# Force each mutant to have its own color
cmap = plt.get_cmap("tab20", len(plot_order))
colors = {
    condition: cmap(i)
    for i, condition in enumerate(plot_order)
}

# Force each mutant to have its own marker shape
marker_shapes = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">", "h", "d"]
markers = {
    condition: marker_shapes[i % len(marker_shapes)]
    for i, condition in enumerate(plot_order)
}

# Plot and save EC50s
df_EC50_RARBRG_exp1 = pd.DataFrame(columns = plot_order,index=[1,2])
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
for condition in plot_order:
    # Save EC50s
    x = df_concentration.loc[condition+"__1"].values
    y = df_signaling.loc[condition+"__1"].values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_1, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([95,1e-13], [105, 1e-11]),p0=[105,1e-12])
    x = df_concentration.loc[condition+"__2"].values
    y = df_signaling.loc[condition+"__2"].values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_2, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([95,1e-13], [105, 1e-11]),p0=[105,1e-12])
    df_EC50_RARBRG_exp1[condition] = [fit_1[1],fit_2[1]]
    
    # Plot
    x = conc_mean.loc[condition].values
    y = sig_mean.loc[condition].values
    yerr = sig_std.loc[condition].values
    ax.errorbar(
        x,
        y/sig_mean.loc['WT'].max()*100,
        yerr=yerr/sig_mean.loc['WT'].max()*100,
        marker=markers[condition],
        linestyle="-",
        label=condition,
        color=colors[condition],
        capsize=7,
        linewidth=3,
        elinewidth=3,
        capthick=3,
        markersize=9,
        markeredgewidth=2,
    )
plt.xlim([5e-15,1e-9])
ax.legend(
    fontsize=17,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0,
)
plt.xscale('log')
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.title("HEK CD25⁺/CD122⁺/CD132⁺", fontsize=25)
plt.ylabel("IL2 signaling (%)", fontsize=25)
plt.xlabel('IL-2 (M)', fontsize=25)
plt.savefig('figures/IL2_mutants/HEK_RARBRG_60_exp1.pdf', transparent=True, bbox_inches="tight")
plt.close()

#### 2nd experiment

In [76]:
df_concentration = pd.read_csv("data/signaling/IL2_mutants/exp2/IL2_concentrations.csv",index_col=0)
data = "HEK_RBRG_60"
df_signaling = pd.read_csv("data/signaling/IL2_mutants/exp2/"+data+".csv",index_col=0)
for mut_name in list(set([mut.split("__")[0] for mut in df_signaling.index])):
    df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]] = df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]]-df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]].mean().min()

# Make sure both dataframes are numeric
df_concentration = df_concentration.apply(pd.to_numeric)
df_signaling = df_signaling.apply(pd.to_numeric)
condition_names = [get_condition_name(idx) for idx in df_signaling.index]

# Mean and std across replicates
conc_mean = df_concentration.groupby(condition_names).mean()
conc_std  = df_concentration.groupby(condition_names).std(ddof=0)
sig_mean = df_signaling.groupby(condition_names).mean()
sig_std  = df_signaling.groupby(condition_names).std(ddof=0)

# Plot order
plot_order = [
    'WT',
    'S130L',
    'S130M',
    'K49R',
    'T123A',
    'N119E',
]

# Plot and save EC50s
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_EC50_RBRG_exp2 = pd.DataFrame(columns = plot_order,index=[1,2,3])
for condition in plot_order:
    # Save EC50s
    x = df_concentration.loc[condition+"__1"].values[2:]
    y = df_signaling.loc[condition+"__1"].values[2:]
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_1, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([97,5e-13], [103, 5e-11]),p0=[98,1e-11])
    x = df_concentration.loc[condition+"__2"].values[2:]
    y = df_signaling.loc[condition+"__2"].values[2:]
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_2, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([97,2e-13], [103, 5e-11]),p0=[98,1e-11])
    x = df_concentration.loc[condition+"__3"].values[2:]
    y = df_signaling.loc[condition+"__3"].values[2:]
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_3, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([97,2e-13], [103, 5e-11]),p0=[98,1e-11])
    df_EC50_RBRG_exp2[condition] = [fit_1[1],fit_2[1],fit_3[1]]
    
    # Plot
    x = conc_mean.loc[condition].to_numpy()
    y = sig_mean.loc[condition].to_numpy()
    xerr = conc_std.loc[condition].to_numpy()
    yerr = sig_std.loc[condition].to_numpy()
    ax.errorbar(
        x,
        y/sig_mean.loc['WT'].max()*100,
        yerr=yerr/sig_mean.loc['WT'].max()*100,
        marker=markers[condition],
        linestyle="-",
        label=condition,
        color=colors[condition],
        capsize=7,
        linewidth=3,
        elinewidth=3,
        capthick=3,
        markersize=9,
        markeredgewidth=2,
    )
    
plt.xlim([5e-15, 1e-9])
ax.legend(
    fontsize=17,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0,
)
plt.xscale('log')
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.title("HEK CD25⁻/CD122⁺/CD132⁺", fontsize=25)
plt.ylabel("IL2 signaling (%)", fontsize=25)
plt.xlabel('IL-2 (M)', fontsize=25)
plt.savefig('figures/IL2_mutants/HEK_RBRG_60_exp2.pdf', transparent=True, bbox_inches="tight")
plt.close()

In [86]:
df_concentration = pd.read_csv("data/signaling/IL2_mutants/exp2/IL2_concentrations.csv",index_col=0)
data = "HEK_RARBRG_60"
df_signaling = pd.read_csv("data/signaling/IL2_mutants/exp2/"+data+".csv",index_col=0)
for mut_name in list(set([mut.split("__")[0] for mut in df_signaling.index])):
    df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]] = df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]]-df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]].mean().min()

# Make sure both dataframes are numeric
df_concentration = df_concentration.apply(pd.to_numeric)
df_signaling = df_signaling.apply(pd.to_numeric)
condition_names = [get_condition_name(idx) for idx in df_signaling.index]

# Mean and std across replicates
conc_mean = df_concentration.groupby(condition_names).mean()
conc_std  = df_concentration.groupby(condition_names).std(ddof=0)
sig_mean = df_signaling.groupby(condition_names).mean()
sig_std  = df_signaling.groupby(condition_names).std(ddof=0)

# Plot order
plot_order = [
    'WT',
    'S130L',
    'S130M',
    'K49R',
    'T123A',
    'N119E',
]

# Plot and save EC50s
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_EC50_RARBRG_exp2 = pd.DataFrame(columns = plot_order,index=[1,2,3])
for condition in plot_order:
    # Save EC50s
    x = df_concentration.loc[condition+"__1"].values
    y = df_signaling.loc[condition+"__1"].values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_1, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([95,1e-13], [105, 1e-11]),p0=[105,1e-11])
    x = df_concentration.loc[condition+"__2"].values
    y = df_signaling.loc[condition+"__2"].values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_2, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([95,1e-13], [105, 1e-11]),p0=[105,1e-11])
    x = df_concentration.loc[condition+"__3"].values
    y = df_signaling.loc[condition+"__3"].values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_3, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([95,1e-13], [105, 5e-11]),p0=[105,1e-11])
    df_EC50_RARBRG_exp2[condition] = [fit_1[1],fit_2[1],fit_3[1]]
    
    # Plot
    x = conc_mean.loc[condition].to_numpy()
    y = sig_mean.loc[condition].to_numpy()
    xerr = conc_std.loc[condition].to_numpy()
    yerr = sig_std.loc[condition].to_numpy()
    ax.errorbar(
        x,
        y/sig_mean.loc['WT'].max()*100,
        yerr=yerr/sig_mean.loc['WT'].max()*100,
        marker=markers[condition],
        linestyle="-",
        label=condition,
        color=colors[condition],
        capsize=7,
        linewidth=3,
        elinewidth=3,
        capthick=3,
        markersize=9,
        markeredgewidth=2,
    )
plt.xlim([5e-15,1e-9])
ax.legend(
    fontsize=17,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0,
)
plt.xscale('log')
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.title("HEK CD25⁺/CD122⁺/CD132⁺", fontsize=25)
plt.ylabel("IL2 signaling (%)", fontsize=25)
plt.xlabel('IL-2 (M)', fontsize=25)
plt.savefig('figures/IL2_mutants/HEK_RARBRG_60_exp2.pdf', transparent=True, bbox_inches="tight")
plt.close()

#### 3rd experiment

In [82]:
df_concentration = pd.read_csv("data/signaling/IL2_mutants/exp3/IL2_concentrations.csv",index_col=0)
data = "HEK_RBRG_60"
df_signaling = pd.read_csv("data/signaling/IL2_mutants/exp3/"+data+".csv",index_col=0)
for mut_name in list(set([mut.split("__")[0] for mut in df_signaling.index])):
    df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]] = df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]]-df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]].mean().min()

# Make sure both dataframes are numeric
df_concentration = df_concentration.apply(pd.to_numeric)
df_signaling = df_signaling.apply(pd.to_numeric)
condition_names = [get_condition_name(idx) for idx in df_signaling.index]

# Mean and std across replicates
conc_mean = df_concentration.groupby(condition_names).mean()
conc_std  = df_concentration.groupby(condition_names).std(ddof=0)
sig_mean = df_signaling.groupby(condition_names).mean()
sig_std  = df_signaling.groupby(condition_names).std(ddof=0)

# Plot order
plot_order = [
    'WT',
    'S130L',
    'S130M',
    'K49R',
    'T123A',
    'N119E',
]

# Plot and save EC50s
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_EC50_RBRG_exp3 = pd.DataFrame(columns = plot_order,index=[1,2,3])
for condition in plot_order:
    # Save EC50s
    x = df_concentration.loc[condition+"__1"].values[2:]
    y = df_signaling.loc[condition+"__1"].values[2:]
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_1, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([97,5e-13], [103, 5e-11]),p0=[98,1e-11])
    x = df_concentration.loc[condition+"__2"].values[2:]
    y = df_signaling.loc[condition+"__2"].values[2:]
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_2, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([97,2e-13], [103, 5e-11]),p0=[98,1e-11])
    x = df_concentration.loc[condition+"__3"].values[2:]
    y = df_signaling.loc[condition+"__3"].values[2:]
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_3, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([97,2e-13], [103, 5e-11]),p0=[98,1e-11])
    df_EC50_RBRG_exp3[condition] = [fit_1[1],fit_2[1],fit_3[1]]
    
    # Plot
    x = conc_mean.loc[condition].to_numpy()
    y = sig_mean.loc[condition].to_numpy()
    xerr = conc_std.loc[condition].to_numpy()
    yerr = sig_std.loc[condition].to_numpy()
    ax.errorbar(
        x,
        y/sig_mean.loc['WT'].max()*100,
        yerr=yerr/sig_mean.loc['WT'].max()*100,
        marker=markers[condition],
        linestyle="-",
        label=condition,
        color=colors[condition],
        capsize=7,
        linewidth=3,
        elinewidth=3,
        capthick=3,
        markersize=9,
        markeredgewidth=2,
    )
    
plt.xlim([5e-15, 1e-9])
ax.legend(
    fontsize=17,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0,
)
plt.xscale('log')
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.title("HEK CD25⁻/CD122⁺/CD132⁺", fontsize=25)
plt.ylabel("IL2 signaling (%)", fontsize=25)
plt.xlabel('IL-2 (M)', fontsize=25)
plt.savefig('figures/IL2_mutants/HEK_RBRG_60_exp3.pdf', transparent=True, bbox_inches="tight")
plt.close()

In [83]:
df_concentration = pd.read_csv("data/signaling/IL2_mutants/exp3/IL2_concentrations.csv",index_col=0)
data = "HEK_RARBRG_60"
df_signaling = pd.read_csv("data/signaling/IL2_mutants/exp3/"+data+".csv",index_col=0)
for mut_name in list(set([mut.split("__")[0] for mut in df_signaling.index])):
    df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]] = df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]]-df_signaling.loc[[mut for mut in df_signaling.index if mut_name+"__" in mut]].mean().min()

# Make sure both dataframes are numeric
df_concentration = df_concentration.apply(pd.to_numeric)
df_signaling = df_signaling.apply(pd.to_numeric)
condition_names = [get_condition_name(idx) for idx in df_signaling.index]

# Mean and std across replicates
conc_mean = df_concentration.groupby(condition_names).mean()
conc_std  = df_concentration.groupby(condition_names).std(ddof=0)
sig_mean = df_signaling.groupby(condition_names).mean()
sig_std  = df_signaling.groupby(condition_names).std(ddof=0)

# Plot order
plot_order = [
    'WT',
    'S130L',
    'S130M',
    'K49R',
    'T123A',
    'N119E',
]

# Plot and save EC50s
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_EC50_RARBRG_exp3 = pd.DataFrame(columns = plot_order,index=[1,2,3])
for condition in plot_order:
    # Save EC50s
    x = df_concentration.loc[condition+"__1"].values[1:]
    y = df_signaling.loc[condition+"__1"].values[1:]
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_1, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([95,1e-13], [105, 1e-11]),p0=[105,1e-12])
    x = df_concentration.loc[condition+"__2"].values[1:]
    y = df_signaling.loc[condition+"__2"].values[1:]
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_2, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([95,1e-13], [105, 1e-11]),p0=[105,1e-12])
    x = df_concentration.loc[condition+"__3"].values[1:]
    y = df_signaling.loc[condition+"__3"].values[1:]
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    fit_3, cov = curve_fit(hill_func, x, y/np.max(y)*100, bounds = ([95,1e-13], [105, 5e-11]),p0=[105,1e-12])
    df_EC50_RARBRG_exp3[condition] = [fit_1[1],fit_2[1],fit_3[1]]
    
    # Plot
    x = conc_mean.loc[condition].to_numpy()
    y = sig_mean.loc[condition].to_numpy()
    xerr = conc_std.loc[condition].to_numpy()
    yerr = sig_std.loc[condition].to_numpy()
    ax.errorbar(
        x,
        y/sig_mean.loc['WT'].max()*100,
        yerr=yerr/sig_mean.loc['WT'].max()*100,
        marker=markers[condition],
        linestyle="-",
        label=condition,
        color=colors[condition],
        capsize=7,
        linewidth=3,
        elinewidth=3,
        capthick=3,
        markersize=9,
        markeredgewidth=2,
    )
    
plt.xlim([5e-15,1e-9])
ax.legend(
    fontsize=17,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0,
)
plt.xscale('log')
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.title("HEK CD25⁺/CD122⁺/CD132⁺", fontsize=25)
plt.ylabel("IL2 signaling (%)", fontsize=25)
plt.xlabel('IL-2 (M)', fontsize=25)
plt.savefig('figures/IL2_mutants/HEK_RARBRG_60_exp3.pdf', transparent=True, bbox_inches="tight")
plt.close()

### Plot ratios EC50s

In [124]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import ttest_ind

exp_markers = {1: "o", 2: "s", 3: "^"}


def stars(p):
    if np.isnan(p): return ""
    if p < 0.0001: return "****"
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return "ns"


def compute_pvalues(df_ratio, ref="WT"):
    """Welch t-test of each mutant vs WT, Bonferroni-adjusted."""
    mutants = [c for c in df_ratio.columns if c != ref]
    n_comp = len(mutants)
    pvals = {}
    for m in mutants:
        res = ttest_ind(
            df_ratio[m].dropna().to_numpy(),
            df_ratio[ref].dropna().to_numpy(),
            equal_var=False,
            nan_policy="omit",
        )
        pvals[m] = min(res.pvalue * n_comp, 1.0)   # Bonferroni
    return pvals


def plot_ec50_ratio(df_ratio, colors, outfile,
                    ylabel="$EC50_{WT}$ / $EC50_{mutant}$", ref="WT", ylim_top=None):
    mutants = df_ratio.columns.tolist()
    pvals = compute_pvalues(df_ratio, ref=ref)
    for m, p in pvals.items():
        print(f"{m}: Bonferroni-adjusted p = {p:.4g} {stars(p)}")

    fig, ax = plt.subplots(1, 1, figsize=(6, 5.5), dpi=400)
    ax.axhline(1, color="grey", linestyle="--", linewidth=2, zorder=1)

    all_values = []
    for i, mutant in enumerate(mutants, start=1):
        c = colors[mutant]
        s = df_ratio[mutant].dropna()
        values = s.to_numpy()
        exps = s.index.get_level_values(0).to_numpy()
        all_values.extend(values)

        ax.errorbar(i, np.mean(values), yerr=np.std(values), fmt="none", color=c,
                    capsize=14, elinewidth=4, capthick=4, zorder=4)
        ax.hlines(np.mean(values), i - 0.425, i + 0.425, color=c, linewidth=7, zorder=3)

        for exp, marker in exp_markers.items():
            mask = exps == exp
            if not mask.any():
                continue
            x = np.random.normal(i, 0.06, size=mask.sum())
            ax.scatter(x, values[mask], facecolor=c, edgecolor="black",
                       linewidth=1.2, s=70, zorder=5, marker=marker)

    # ---- Significance brackets (only significant comparisons, p < 0.05) ----
    wt_pos = mutants.index(ref) + 1
    y0 = max(all_values) * 1.05
    step = max(all_values) * 0.12
    h = step * 0.15

    sig_pvals = {m: p for m, p in pvals.items() if p < 0.05}   # drops "ns"

    for idx, (mutant, p) in enumerate(sig_pvals.items()):
        mut_pos = mutants.index(mutant) + 1
        y = y0 + idx * step                                    # stack only the drawn brackets
        ax.plot([wt_pos, wt_pos, mut_pos, mut_pos], [y, y + h, y + h, y],
                color="black", linewidth=2)
        ax.text((wt_pos + mut_pos) / 2, y - h * 0.6, stars(p),
                ha="center", va="bottom", fontsize=22)

    top = ylim_top if ylim_top is not None else y0 + len(sig_pvals) * step + step
    ax.set_ylim(bottom=0, top=top)

    # Formatting
    for sp in ("bottom", "left"):
        ax.spines[sp].set_linewidth(4)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.xaxis.set_tick_params(width=4, length=8)
    ax.yaxis.set_tick_params(width=4, length=8)
    ax.tick_params(axis="both", labelsize=16)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_xticks(range(1, len(mutants) + 1))
    ax.set_xticklabels(mutants, fontsize=16)

    handles = [Line2D([0], [0], marker=m, linestyle="none", markerfacecolor="lightgrey",
                      markeredgecolor="black", markersize=9, label=f"Exp {e}")
               for e, m in exp_markers.items()]
    ax.legend(handles=handles, fontsize=10, loc="upper left")

    plt.tight_layout()
    plt.savefig(outfile, transparent=True, bbox_inches="tight")
    plt.close(fig)
    return pvals


# ---- HEK RARBRG ----
df_EC50_RARBRG_exp1_plot = df_EC50_RARBRG_exp1 / df_EC50_RARBRG_exp1["WT"].mean()
df_EC50_RARBRG_exp2_plot = df_EC50_RARBRG_exp2 / df_EC50_RARBRG_exp2["WT"].mean()
df_EC50_RARBRG_exp3_plot = df_EC50_RARBRG_exp3 / df_EC50_RARBRG_exp3["WT"].mean()
df_EC50_RARBRG_exp1_plot = df_EC50_RARBRG_exp1_plot[df_EC50_RARBRG_exp2_plot.columns]
df_ratio_RARBRG = pd.concat(
    [df_EC50_RARBRG_exp1_plot, df_EC50_RARBRG_exp2_plot, df_EC50_RARBRG_exp3_plot],
    keys=[1, 2, 3])
p_RARBRG = plot_ec50_ratio(df_ratio_RARBRG, colors, "figures/IL2_mutants/Plot_EC50_ratio_RARBRG.pdf")

# ---- HEK RBRG ----
df_EC50_RBRG_exp1_plot = df_EC50_RBRG_exp1 / df_EC50_RBRG_exp1["WT"].mean()
df_EC50_RBRG_exp2_plot = df_EC50_RBRG_exp2 / df_EC50_RBRG_exp2["WT"].mean()
df_EC50_RBRG_exp3_plot = df_EC50_RBRG_exp3 / df_EC50_RBRG_exp3["WT"].mean()
df_EC50_RBRG_exp1_plot = df_EC50_RBRG_exp1_plot[df_EC50_RBRG_exp2_plot.columns]
df_ratio_RBRG = pd.concat(
    [df_EC50_RBRG_exp1_plot, df_EC50_RBRG_exp2_plot, df_EC50_RBRG_exp3_plot],
    keys=[1, 2, 3])
p_RBRG = plot_ec50_ratio(df_ratio_RBRG, colors, "figures/IL2_mutants/Plot_EC50_ratio_RBRG.pdf")

S130L: Bonferroni-adjusted p = 1 ns
S130M: Bonferroni-adjusted p = 1 ns
K49R: Bonferroni-adjusted p = 1 ns
T123A: Bonferroni-adjusted p = 1 ns
N119E: Bonferroni-adjusted p = 1 ns
S130L: Bonferroni-adjusted p = 1 ns
S130M: Bonferroni-adjusted p = 0.09227 ns
K49R: Bonferroni-adjusted p = 0.0007761 ***
T123A: Bonferroni-adjusted p = 0.1364 ns
N119E: Bonferroni-adjusted p = 0.0001612 ***
S130L: Bonferroni-adjusted p = 0.03461 *
S130M: Bonferroni-adjusted p = 2.907e-07 ****
K49R: Bonferroni-adjusted p = 1.754e-05 ****
T123A: Bonferroni-adjusted p = 0.5633 ns
N119E: Bonferroni-adjusted p = 1.268e-07 ****


#### Integrate the data from both experiments and get a plot to demonstrate whihc mutants increase the specificity towards cells that express IL-2RB and IL-2RG

In [145]:
# ---- HEK RARBRG ----
df_EC50_RARBRG_exp1_plot = np.log10(df_EC50_RARBRG_exp1)-np.log10(df_EC50_RARBRG_exp1)["WT"].mean()
df_EC50_RARBRG_exp2_plot = np.log10(df_EC50_RARBRG_exp2)-np.log10(df_EC50_RARBRG_exp2)["WT"].mean()
df_EC50_RARBRG_exp3_plot = np.log10(df_EC50_RARBRG_exp3)-np.log10(df_EC50_RARBRG_exp3)["WT"].mean()
df_EC50_RARBRG_exp1_plot = df_EC50_RARBRG_exp1_plot[df_EC50_RARBRG_exp2_plot.columns]

# ---- HEK RBRG ----
df_EC50_RBRG_exp1_plot = np.log10(df_EC50_RBRG_exp1)-np.log10(df_EC50_RBRG_exp1)["WT"].mean()
df_EC50_RBRG_exp2_plot = np.log10(df_EC50_RBRG_exp2)-np.log10(df_EC50_RBRG_exp2)["WT"].mean()
df_EC50_RBRG_exp3_plot = np.log10(df_EC50_RBRG_exp3)-np.log10(df_EC50_RBRG_exp3)["WT"].mean()
df_EC50_RBRG_exp1_plot = df_EC50_RBRG_exp1_plot[df_EC50_RBRG_exp2_plot.columns]

# Get ratios (deltadeltaEC50)
df_ratio_RARBRG_RBRG = pd.concat([df_EC50_RARBRG_exp1_plot-df_EC50_RBRG_exp1_plot,df_EC50_RARBRG_exp2_plot-df_EC50_RBRG_exp2_plot,df_EC50_RARBRG_exp3_plot-df_EC50_RBRG_exp3_plot],keys=[1, 2, 3])

In [164]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import ttest_ind

exp_markers = {1: "o", 2: "s", 3: "^"}


def stars(p):
    if np.isnan(p): return ""
    if p < 0.0001: return "****"
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return "ns"


def dEC50(df):
    """log10(EC50) relative to the mean log10(EC50) of WT in that experiment/cell type."""
    l = np.log10(df)
    return l - l["WT"].mean()


def ddEC50(df_RARBRG_exp, df_target_exp):
    """ddEC50 = dEC50(RARBRG) - dEC50(target cell). Positive = more selective for target cells."""
    a, b = dEC50(df_RARBRG_exp), dEC50(df_target_exp)
    cols = [c for c in a.columns if c in b.columns]
    return a[cols] - b[cols]          # replicate i of one cell type paired with replicate i of the other


def compute_pvalues(df, ref="WT"):
    """One-sided Welch t-test (mutant ddEC50 > WT ddEC50), Bonferroni-adjusted."""
    mutants = [c for c in df.columns if c != ref]
    n_comp = len(mutants)
    pvals = {}
    for m in mutants:
        res = ttest_ind(
            df[m].dropna().to_numpy(),
            df[ref].dropna().to_numpy(),
            equal_var=False,
            alternative="greater",
            nan_policy="omit",
        )
        pvals[m] = res.pvalue
    return pvals


def plot_ddEC50(df, colors, outfile, ylabel, ref="WT", ylim_top=None):
    mutants = df.columns.tolist()
    pvals = compute_pvalues(df, ref=ref)
    for m, p in pvals.items():
        print(f"{m}: One-sided p = {p:.4g} {stars(p)}")

    fig, ax = plt.subplots(1, 1, figsize=(6, 5.5), dpi=400)
    ax.axhline(0, color="grey", linestyle="--", linewidth=2, zorder=1)

    all_values = []
    for i, mutant in enumerate(mutants, start=1):
        c = colors[mutant]
        s = df[mutant].dropna()
        values = s.to_numpy()
        exps = s.index.get_level_values(0).to_numpy()
        all_values.extend(values)

        ax.errorbar(i, np.mean(values), yerr=np.std(values), fmt="none", color=c,
                    capsize=14, elinewidth=4, capthick=4, zorder=4)
        ax.hlines(np.mean(values), i - 0.425, i + 0.425, color=c, linewidth=7, zorder=3)

        for exp, marker in exp_markers.items():
            mask = exps == exp
            if not mask.any():
                continue
            x = np.random.normal(i, 0.06, size=mask.sum())
            ax.scatter(x, values[mask], facecolor=c, edgecolor="black",
                       linewidth=1.2, s=70, zorder=5, marker=marker)

    # ---- Significance brackets (only significant, p < 0.05) ----
    wt_pos = mutants.index(ref) + 1
    vmax, vmin = max(all_values), min(all_values)
    rng = max(vmax - min(vmin, 0), 1e-9)
    y0 = vmax + 0.06 * rng
    step = 0.12 * rng
    h = step * 0.15

    sig_pvals = {m: p for m, p in pvals.items() if p < 0.05}
    for idx, (mutant, p) in enumerate(sig_pvals.items()):
        mut_pos = mutants.index(mutant) + 1
        y = y0 + idx * step
        ax.plot([wt_pos, wt_pos, mut_pos, mut_pos], [y, y + h, y + h, y],
                color="black", linewidth=2)
        ax.text((wt_pos + mut_pos) / 2, y - h * 0.6, stars(p),
                ha="center", va="bottom", fontsize=22)

    top = ylim_top if ylim_top is not None else y0 + len(sig_pvals) * step + step
    ax.set_ylim(bottom=min(vmin, 0) - 0.08 * rng, top=top)

    # Formatting
    for sp in ("bottom", "left"):
        ax.spines[sp].set_linewidth(4)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.xaxis.set_tick_params(width=4, length=8)
    ax.yaxis.set_tick_params(width=4, length=8)
    ax.tick_params(axis="both", labelsize=16)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_xticks(range(1, len(mutants) + 1))
    ax.set_xticklabels(mutants, fontsize=16)

    handles = [Line2D([0], [0], marker=m, linestyle="none", markerfacecolor="lightgrey",
                      markeredgecolor="black", markersize=9, label=f"Exp {e}")
               for e, m in exp_markers.items()]
    ax.legend(handles=handles, fontsize=10, loc="upper left")

    plt.tight_layout()
    plt.savefig(outfile, transparent=True, bbox_inches="tight")
    plt.close(fig)
    return pvals


# ---- ddEC50: RARBRG vs RBRG (experiments 1-3) ----
p_dd_RBRG = plot_ddEC50(
    df_ratio_RARBRG_RBRG, colors, "figures/IL2_mutants/Plot_ddEC50_RARBRG_vs_RBRG.pdf",
    ylabel="$\\Delta\\Delta$EC50 RARBRG vs RBRG (log10 M)")

S130L: One-sided p = 0.2376 ns
S130M: One-sided p = 0.1423 ns
K49R: One-sided p = 0.03011 *
T123A: One-sided p = 0.07309 ns
N119E: One-sided p = 0.008948 **
S130L: One-sided p = 0.3837 ns
S130M: One-sided p = 0.2728 ns
K49R: One-sided p = 0.07708 ns
T123A: One-sided p = 0.1965 ns
N119E: One-sided p = 0.03782 *
